# 03 Model comparisons

**Question: does one forecast give a better portfolio than another?** This is the research
question. For each pair (a, b) in `COMPARISONS` in `config.py`, the Ledoit-Wolf (2008) test of
H₀: SR(a) = SR(b) is run on the two managed portfolios directly, for the cyclical bucket and for
each ETF on its own, under both specifications of σ̄².

**Why a direct test.** Notebook 02 tests each model against buy-and-hold. One model beating BH
significantly while another does not is not a test of the difference between them: the two
managed portfolios are highly correlated, so the gap between them can be small even when their
p-values against BH fall on opposite sides of 5%. The Ledoit-Wolf test uses that correlation.

**No control is needed.** Within a specification, both models in a pair use the same σ̄²
(fixed, or the same σ̄²ᵢ,ₜ), so the difference between their portfolios comes only from the
forecasts. Running every pair under both specifications shows whether the conclusion depends on
the choice of reference.

## Setup

In [1]:
import json
import numpy as np
import pandas as pd

from config import COMPARISONS
from portfolio_utils import PORTFOLIO_DIR, EVAL_DIR, load_portfolios, compare_pair

pd.set_option("display.width", 140)
pd.set_option("display.max_columns", 30)
pd.set_option("display.float_format", "{:.4f}".format)

In [2]:
run_info = json.loads((PORTFOLIO_DIR / "run_info.json").read_text())
MODELS, SPECS, K_CAP = run_info["variants"], run_info["specs"], run_info["k_cap"]

portfolios = {}
for spec in SPECS:
    portfolios[spec], xret = load_portfolios(spec)
    assert set(MODELS) <= set(portfolios[spec]), f"{spec}: saved models do not match run_info"
TICKERS = list(xret.columns)
bh = xret.mean(axis=1)

print("Models:", MODELS)
print("Specifications:", SPECS)
print(f"Out-of-sample window: {run_info['oos_window'][0]} to {run_info['oos_window'][1]}"
      f" ({len(xret)} months)")

missing = {m for pair in COMPARISONS for m in pair} - set(MODELS)
assert not missing, f"COMPARISONS uses models not built in 01_build_portfolios: {sorted(missing)}"
print("Pairs:", COMPARISONS)

Models: ['garch', 'garch_midas_baa10y', 'ols_1m', 'en_full_macro', 'en_no_macro', 'en_no_vix', 'xgb_full_macro', 'xgb_no_macro', 'xgb_no_vix', 'xgb_vix_only']
Specifications: ['fixed', 'expanding']
Out-of-sample window: 2008-03 to 2025-12 (214 months)
Pairs: [('en_full_macro', 'en_no_macro'), ('xgb_full_macro', 'xgb_no_macro'), ('en_full_macro', 'en_no_vix'), ('xgb_full_macro', 'xgb_no_vix'), ('en_full_macro', 'garch'), ('xgb_full_macro', 'garch')]


## Ledoit-Wolf tests, every pair and specification

For each pair: SR_a and SR_b (annualised), SR_diff = SR_a − SR_b, the studentised statistic LW_t
and the bootstrap p-value LW_p. Positive SR_diff means model a gives the better portfolio. The
per-ETF tests use each ETF's managed series rescaled by cᵢ, which does not change its Sharpe ratio.

In [3]:
model_pairs = pd.concat(
    {(spec, a, b): compare_pair(portfolios[spec][a], portfolios[spec][b], xret)
     for spec in SPECS for a, b in COMPARISONS},
    names=["spec", "model_a", "model_b"])
model_pairs = model_pairs[["SR_a", "SR_b", "SR_diff", "LW_t", "LW_p"]].astype(float)

### Cyclical bucket

One row per pair, with the two specifications side by side.

In [4]:
stats = ["SR_a", "SR_b", "SR_diff", "LW_t", "LW_p"]
pair_idx = pd.MultiIndex.from_tuples(COMPARISONS, names=["model_a", "model_b"])
bucket_pairs = (model_pairs.xs("Cyclical", level="ETF").unstack("spec")
                .swaplevel(axis=1)
                .reindex(columns=pd.MultiIndex.from_product([SPECS, stats]))
                .reindex(pair_idx))
bucket_pairs

fixed                               expanding                              
                              SR_a   SR_b SR_diff    LW_t   LW_p      SR_a   SR_b SR_diff    LW_t   LW_p
model_a        model_b                                                                                  
en_full_macro  en_no_macro  0.3990 0.4536 -0.0547 -1.3641 0.1786    0.5391 0.5926 -0.0536 -1.6321 0.1204
xgb_full_macro xgb_no_macro 0.3375 0.4284 -0.0911 -1.8363 0.0766    0.4601 0.5660 -0.1061 -2.7542 0.0130
en_full_macro  en_no_vix    0.3990 0.4161 -0.0171 -0.4073 0.7033    0.5391 0.5497 -0.0106 -0.2570 0.8108
xgb_full_macro xgb_no_vix   0.3375 0.3982 -0.0609 -1.2396 0.2629    0.4601 0.5110 -0.0510 -1.4302 0.1860
en_full_macro  garch        0.3990 0.4966 -0.0978 -1.9280 0.0768    0.5391 0.6079 -0.0689 -1.7846 0.0976
xgb_full_macro garch        0.3375 0.4966 -0.1595 -2.1351 0.0836    0.4601 0.6079 -0.1481 -2.3891 0.0776

### Per ETF

SR_diff and LW_p for each pair (rows) and ETF (columns), one table per specification.

In [5]:
per_etf_pairs = {}
for spec in SPECS:
    sub = model_pairs.loc[spec].drop(index="Cyclical", level="ETF")
    per_etf_pairs[spec] = pd.concat(
        {"SR_diff": sub["SR_diff"].unstack("ETF")[TICKERS],
         "LW_p": sub["LW_p"].unstack("ETF")[TICKERS]}, axis=1).reindex(pair_idx)
    print(f"--- {spec} σ̄² ---")
    display(per_etf_pairs[spec])

--- fixed σ̄² ---


SR_diff                                   LW_p                            
ETF                             XLB     XLE     XLF     XLI     XLY    XLB    XLE    XLF    XLI    XLY
model_a        model_b                                                                                
en_full_macro  en_no_macro  -0.0713 -0.0194 -0.0467 -0.0736 -0.0650 0.1792 0.6453 0.5233 0.2703 0.1946
xgb_full_macro xgb_no_macro -0.0891 -0.0461 -0.0068 -0.0975 -0.1517 0.1092 0.3391 0.9356 0.2537 0.1006
en_full_macro  en_no_vix    -0.0463 -0.0063 -0.0157  0.0049  0.0047 0.3365 0.4935 0.8078 0.9382 0.9088
xgb_full_macro xgb_no_vix   -0.1109  0.0010  0.0010 -0.0834 -0.0976 0.0516 0.9684 0.9902 0.2066 0.1412
en_full_macro  garch        -0.1053 -0.0390 -0.1270 -0.1456 -0.0800 0.0666 0.5325 0.1054 0.0374 0.1760
xgb_full_macro garch        -0.1321 -0.1162 -0.1349 -0.1844 -0.1408 0.0576 0.0978 0.1874 0.0862 0.1872

--- expanding σ̄² ---


SR_diff                                   LW_p                            
ETF                             XLB     XLE     XLF     XLI     XLY    XLB    XLE    XLF    XLI    XLY
model_a        model_b                                                                                
en_full_macro  en_no_macro  -0.0455 -0.0340 -0.0613 -0.0670 -0.0615 0.3141 0.2913 0.1392 0.2228 0.1872
xgb_full_macro xgb_no_macro -0.0522 -0.0536 -0.1017 -0.1308 -0.0859 0.3021 0.1600 0.0650 0.0418 0.2366
en_full_macro  en_no_vix    -0.0288 -0.0114 -0.0121  0.0121 -0.0025 0.5021 0.1742 0.8698 0.8486 0.9550
xgb_full_macro xgb_no_vix   -0.0764  0.0128 -0.0480 -0.0883 -0.0657 0.0928 0.5599 0.2392 0.1220 0.2266
en_full_macro  garch        -0.0642 -0.0579 -0.0551 -0.0783 -0.0594 0.1252 0.2589 0.3503 0.1088 0.2811
xgb_full_macro garch        -0.0994 -0.1132 -0.1401 -0.1631 -0.1174 0.0946 0.0556 0.1600 0.0404 0.1126

## Save the tables

Written to `results/eval/`:

- `model_pairs_bucket.csv`: the cyclical-bucket table (pairs x specification);
- `model_pairs_per_etf.csv`: every test, long (spec, model_a, model_b, ETF), including the
  cyclical bucket.

In [6]:
EVAL_DIR.mkdir(parents=True, exist_ok=True)
bucket_pairs.to_csv(EVAL_DIR / "model_pairs_bucket.csv")
model_pairs.to_csv(EVAL_DIR / "model_pairs_per_etf.csv")
print("Saved to", EVAL_DIR.resolve())

Saved to C:\Documents\NUS\Y4S1\DSE4101\dse4101capstone_project\scripts\portfolio_construction\results\eval
